# Notebook 01 – Preprocessing Time-Series Data
IITB Project – Problem ID-9: Effect of Feedback on Affective and Cognitive Measures

## Goals of This Notebook
1. Load and parse PSY.csv, TIVA.csv, EEG.csv files for each participant (1–38).
2. Synchronize timestamps (trial start/end) to slice TIVA and EEG signals per trial.
3. Extract trial-level features:
   - Affective states → mean engagement & confusion
   - EEG signals → compute band powers (delta, theta, alpha, beta, gamma)
4. Merge into a per-trial dataset across all participants.
5. Save results to:
   - preprocessed_features.csv (trial-level features)
   - feedback_summary.csv (feedback counts per participant)

## Input Data
- PSY.csv → trial timing & feedback type  
- TIVA.csv → affective state data  
- EEG.csv → brainwave recordings  

## Output
- preprocessed_features.csv → dataset used in downstream ML/DL modeling  
- feedback_summary.csv → counts of correct/incorrect feedback per participant


In [1]:
# 01_preprocessing_time_series.ipynb
# ==================================
# IITB Project – Problem ID-9
# Time-series preprocessing for PSY, TIVA, EEG data

import warnings
warnings.filterwarnings("ignore")            # hide runtime warnings
import numpy as np
np.seterr(all="ignore")                      # ignore numpy floating errors during FFT

import pandas as pd
from pathlib import Path

# -------------------------------
# Config
# -------------------------------
DATASET_ROOT = Path("../data/STData")       # adjust if your data is elsewhere
SUBSET = list(range(1, 39))                 # participants 1..38
OUT_DIR = Path("../results")
OUT_PATH = OUT_DIR / "preprocessed_features.csv"
FEEDBACK_SUM_PATH = OUT_DIR / "feedback_summary.csv"

MIN_EEG_SAMPLES = 50        # minimum samples for FFT
DEFAULT_SFREQ = 250         # assumed EEG sampling frequency

# Candidate column name lists (robust lookup)
PSY_START_NAMES = ["routineStart", "RoutineStart", "start", "StartTime", "Start"]
PSY_END_NAMES   = ["routineEnd", "RoutineEnd", "end", "EndTime", "End"]
PSY_TRIALKEY    = ["Key", "trial_id", "QuestionKey", "Trial", "TrialID"]
PSY_FEEDBACK    = ["verdict", "feedback", "Feedback", "Verdict"]

TIVA_TIME_NAMES = ["UnixTime", "time", "Timestamp", "Time", "timestamp"]
TIVA_ENG_NAMES  = ["Engagement", "engagement", "ENGAGEMENT"]
TIVA_CONF_NAMES = ["Confusion", "confusion", "CONFUSION"]

# -------------------------------
# Helpers
# -------------------------------
def find_first_column(df_cols, candidates):
    """Return first matching column name from candidates."""
    for c in candidates:
        if c in df_cols:
            return c
    return None

def safe_get(row, names, default=np.nan):
    """Try multiple names in a row (Series)."""
    for n in names:
        if n in row.index:
            v = row.get(n)
            if pd.notna(v):
                return v
    return default

def compute_band_powers(eeg_df, sfreq=DEFAULT_SFREQ):
    """
    Compute mean power in EEG bands for numeric columns.
    Skips invalid/short signals.
    """
    band_features = {}
    if eeg_df is None or eeg_df.shape[1] == 0:
        return band_features

    for col in eeg_df.columns:
        try:
            signal = eeg_df[col].dropna().values.astype(float)
            if signal.size < MIN_EEG_SAMPLES:
                continue

            freqs = np.fft.rfftfreq(len(signal), 1.0 / sfreq)
            psd = np.abs(np.fft.rfft(signal))**2
            bands = {
                "delta": (0.5, 4),
                "theta": (4, 8),
                "alpha": (8, 12),
                "beta":  (12, 30),
                "gamma": (30, 45)
            }
            for bname, (fmin, fmax) in bands.items():
                mask = (freqs >= fmin) & (freqs <= fmax)
                if mask.any():
                    band_features[f"{col}_{bname}"] = np.nanmean(psd[mask])
        except Exception:
            continue
    return band_features

# -------------------------------
# Participant processing
# -------------------------------
def process_participant(pid):
    """Process one participant folder: PSY, TIVA, EEG → trial-level features."""
    psy_path = DATASET_ROOT / str(pid) / f"{pid}_PSY.csv"
    tiva_path = DATASET_ROOT / str(pid) / f"{pid}_TIVA.csv"
    eeg_path = DATASET_ROOT / str(pid) / f"{pid}_EEG.csv"

    # Load safely
    try:
        psy = pd.read_csv(psy_path, low_memory=False)
    except Exception as e:
        print(f"⚠️  Participant {pid}: cannot open PSY ({psy_path.name}) → {e}")
        return pd.DataFrame()

    try:
        tiva = pd.read_csv(tiva_path, low_memory=False)
    except Exception:
        tiva = pd.DataFrame()

    try:
        eeg = pd.read_csv(eeg_path, low_memory=False)
    except Exception:
        eeg = pd.DataFrame()

    # Column names
    psy_cols = list(psy.columns)
    start_col = find_first_column(psy_cols, PSY_START_NAMES)
    end_col   = find_first_column(psy_cols, PSY_END_NAMES)
    trialkey_col = find_first_column(psy_cols, PSY_TRIALKEY)
    feedback_col = find_first_column(psy_cols, PSY_FEEDBACK)

    tiva_cols = list(tiva.columns) if not tiva.empty else []
    tiva_time_col = find_first_column(tiva_cols, TIVA_TIME_NAMES)
    eng_col = find_first_column(tiva_cols, TIVA_ENG_NAMES)
    conf_col = find_first_column(tiva_cols, TIVA_CONF_NAMES)

    eeg_numeric = eeg.select_dtypes(include=[np.number])
    drop_patterns = ["UnixTime", "Battery", "SampleNumber", "BlinkRate", "Row"]
    eeg_numeric = eeg_numeric[[c for c in eeg_numeric.columns if not any(p in c for p in drop_patterns)]]

    rows = []
    for _, trial_row in psy.iterrows():
        trial_id = safe_get(trial_row, [trialkey_col] if trialkey_col else PSY_TRIALKEY)
        feedback = safe_get(trial_row, [feedback_col] if feedback_col else PSY_FEEDBACK)
        fb_time = trial_row.get("Cat2FeedbackTime", np.nan)

        mean_engagement, mean_confusion = np.nan, np.nan
        if tiva_time_col and eng_col and conf_col:
            start = safe_get(trial_row, [start_col])
            end = safe_get(trial_row, [end_col])
            if pd.notna(start) and pd.notna(end):
                try:
                    tiva_slice = tiva[(tiva[tiva_time_col] >= start) & (tiva[tiva_time_col] <= end)]
                    if not tiva_slice.empty:
                        mean_engagement = tiva_slice[eng_col].astype(float).mean()
                        mean_confusion = tiva_slice[conf_col].astype(float).mean()
                except Exception:
                    pass

        eeg_features = compute_band_powers(eeg_numeric) if not eeg_numeric.empty else {}

        trial_features = {
            "participant": pid,
            "trial_id": trial_id,
            "feedback": feedback,
            "feedback_time": fb_time,
            "engagement": mean_engagement,
            "confusion": mean_confusion
        }
        trial_features.update(eeg_features)
        rows.append(trial_features)

    df_trials = pd.DataFrame(rows)

    if not df_trials.empty:
        counts = df_trials["feedback"].value_counts(dropna=True).to_dict()
        print(f"   → P{pid}: trials={len(df_trials)}, feedback_counts={counts}")
    else:
        print(f"   → P{pid}: NO trials extracted")

    return df_trials

# -------------------------------
# Run for all participants & save
# -------------------------------
all_parts = [process_participant(pid) for pid in SUBSET]
all_parts = [dfp for dfp in all_parts if not dfp.empty]

if all_parts:
    final_df = pd.concat(all_parts, ignore_index=True)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(OUT_PATH, index=False)

    fb_summary = final_df.groupby("participant")["feedback"].value_counts().unstack(fill_value=0)
    fb_summary.to_csv(FEEDBACK_SUM_PATH, index=True)

    saved = pd.read_csv(OUT_PATH, low_memory=False)
    print("\n🎯 Preprocessing finished and saved.")
    print("📂 Saved file:", OUT_PATH.resolve())
    print("📊 Saved shape:", saved.shape)
    print(saved.head(5))
    print("\n📂 Feedback summary saved to:", FEEDBACK_SUM_PATH.resolve())
    print(fb_summary.head(10))
else:
    print("❌ No participant data processed. Check paths and filenames.")


   → P1: trials=40, feedback_counts={'CORRECT': 32, 'INCORRECT': 7}
   → P2: trials=33, feedback_counts={'CORRECT': 22, 'INCORRECT': 10}
   → P3: trials=42, feedback_counts={'CORRECT': 29, 'INCORRECT': 6, 'SKIP': 6}
   → P4: trials=42, feedback_counts={'CORRECT': 25, 'INCORRECT': 13, 'SKIP': 3}
   → P5: trials=44, feedback_counts={'CORRECT': 39, 'INCORRECT': 4, 'SKIP': 1}
   → P6: trials=20, feedback_counts={'CORRECT': 13, 'INCORRECT': 4, 'SKIP': 1}
   → P7: trials=35, feedback_counts={'CORRECT': 23, 'INCORRECT': 11}
   → P8: trials=42, feedback_counts={'CORRECT': 30, 'INCORRECT': 9, 'SKIP': 2}
   → P9: trials=38, feedback_counts={'CORRECT': 29, 'INCORRECT': 8}
   → P10: trials=38, feedback_counts={'INCORRECT': 22, 'CORRECT': 15}
   → P11: trials=44, feedback_counts={'CORRECT': 30, 'INCORRECT': 14}
   → P12: trials=45, feedback_counts={'CORRECT': 29, 'INCORRECT': 16}
   → P13: trials=42, feedback_counts={'CORRECT': 33, 'INCORRECT': 7}
   → P14: trials=33, feedback_counts={'CORRECT': 29